In [ ]:
!pip -q install -U \
  pandas==2.2.2 \
  pyarrow==15.0.2 \
  transformers==4.41.2 \
  datasets==2.19.2 \
  accelerate==0.30.1 \
  peft==0.11.1 \
  trl==0.9.4 \
  bitsandbytes==0.43.1 \
  triton==3.1.0 \
  sentencepiece \
  scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import json
import random
import re
from tqdm.auto import tqdm

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

from datasets import Dataset
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig

# Set seeds so results are more reproducible
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
DATA_PATH = "impostor_pair_dataset.csv"

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (116, 13)


,source,q1_category,q2_category,q1_answers_json,q2_answers_json,shared_answers_json,q1_only_answers_json,q2_only_answers_json,n_q1_answers,n_q2_answers,n_shared,n_q1_only,n_q2_only
0,banks_connell,animal,bird,"[""cat"", ""dog"", ""elephant"", ""giraffe"", ""lion"", ...","[""pigeon"", ""blue tit"", ""duck"", ""blackbird"", ""b...","[""chicken"", ""duck"", ""penguin"", ""flamingo"", ""sw...","[""cat"", ""dog"", ""elephant"", ""giraffe"", ""lion"", ...","[""blue tit"", ""blackbird"", ""bluebird"", ""crow"", ...",69,38,9,60,29
1,banks_connell,animal,farm animal,"[""cat"", ""dog"", ""elephant"", ""giraffe"", ""lion"", ...","[""cow"", ""sheep"", ""pig"", ""chicken"", ""horse"", ""g...","[""cat"", ""dog"", ""horse"", ""cow"", ""pig"", ""rabbit""...","[""elephant"", ""giraffe"", ""lion"", ""tiger"", ""fish...","[""geese"", ""sheepdog"", ""turkey"", ""bull"", ""hen"",...",69,18,11,58,7
2,banks_connell,animal,meat,"[""cat"", ""dog"", ""elephant"", ""giraffe"", ""lion"", ...","[""chicken"", ""beef"", ""pork"", ""lamb"", ""turkey"", ...","[""horse"", ""rabbit"", ""fish"", ""chicken"", ""duck"",...","[""cat"", ""dog"", ""elephant"", ""giraffe"", ""lion"", ...","[""beef"", ""pork"", ""lamb"", ""turkey"", ""veal"", ""ve...",69,26,9,60,17
3,banks_connell,animal,water bird,"[""cat"", ""dog"", ""elephant"", ""giraffe"", ""lion"", ...","[""duck"", ""swan"", ""goose"", ""pelican"", ""penguin""...","[""duck"", ""penguin"", ""flamingo"", ""swan"", ""goose...","[""cat"", ""dog"", ""elephant"", ""giraffe"", ""lion"", ...","[""pelican"", ""heron"", ""kingfisher"", ""canada goo...",69,15,6,63,9
4,banks_connell,bathroom fixture,living room furniture,"[""bath"", ""shower"", ""sink"", ""toilet"", ""mirror"",...","[""sofa"", ""television"", ""chair"", ""coffee table""...","[""mirror"", ""cupboard""]","[""bath"", ""shower"", ""sink"", ""toilet"", ""tap"", ""t...","[""sofa"", ""television"", ""chair"", ""coffee table""...",19,23,2,17,21


In [ ]:
JSON_COLUMNS = [
    "q1_answers_json",
    "q2_answers_json",
    "shared_answers_json",
    "q1_only_answers_json",
    "q2_only_answers_json",
]

def parse_json_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    return json.loads(x)

for col in JSON_COLUMNS:
    df[col] = df[col].apply(parse_json_list)

print("Finished parsing JSON columns.")
display(df.head(2))

Finished parsing JSON columns.


,source,q1_category,q2_category,q1_answers_json,q2_answers_json,shared_answers_json,q1_only_answers_json,q2_only_answers_json,n_q1_answers,n_q2_answers,n_shared,n_q1_only,n_q2_only
0,banks_connell,animal,bird,"[cat, dog, elephant, giraffe, lion, horse, cow...","[pigeon, blue tit, duck, blackbird, bluebird, ...","[chicken, duck, penguin, flamingo, swan, eagle...","[cat, dog, elephant, giraffe, lion, horse, cow...","[blue tit, blackbird, bluebird, crow, parrot, ...",69,38,9,60,29
1,banks_connell,animal,farm animal,"[cat, dog, elephant, giraffe, lion, horse, cow...","[cow, sheep, pig, chicken, horse, goat, duck, ...","[cat, dog, horse, cow, pig, rabbit, sheep, chi...","[elephant, giraffe, lion, tiger, fish, guinea ...","[geese, sheepdog, turkey, bull, hen, lamb, llama]",69,18,11,58,7


In [ ]:
row = df.iloc[0]

print("Q1 category:", row["q1_category"])
print("Q2 category:", row["q2_category"])
print("Q1 answers sample:", row["q1_answers_json"][:5])
print("Q2 answers sample:", row["q2_answers_json"][:5])
print("Shared answers sample:", row["shared_answers_json"][:5])

Q1 category: animal
Q2 category: bird
Q1 answers sample: ['cat', 'dog', 'elephant', 'giraffe', 'lion']
Q2 answers sample: ['pigeon', 'blue tit', 'duck', 'blackbird', 'bluebird']
Shared answers sample: ['chicken', 'duck', 'penguin', 'flamingo', 'swan']


In [ ]:
def normalize_text(text):
    """
    Lowercase and remove extra whitespace/punctuation
    so answer comparisons are more stable.
    """
    text = str(text).strip().lower()
    text = re.sub(r"[\"'`]", "", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

def normalize_list(items):
    return [normalize_text(x) for x in items]

# Create normalized versions of all answer lists
for col in JSON_COLUMNS:
    norm_col = col.replace("_json", "_norm")
    df[norm_col] = df[col].apply(normalize_list)

display(df.head(2))

,source,q1_category,q2_category,q1_answers_json,q2_answers_json,shared_answers_json,q1_only_answers_json,q2_only_answers_json,n_q1_answers,n_q2_answers,n_shared,n_q1_only,n_q2_only,q1_answers_norm,q2_answers_norm,shared_answers_norm,q1_only_answers_norm,q2_only_answers_norm
0,banks_connell,animal,bird,"[cat, dog, elephant, giraffe, lion, horse, cow...","[pigeon, blue tit, duck, blackbird, bluebird, ...","[chicken, duck, penguin, flamingo, swan, eagle...","[cat, dog, elephant, giraffe, lion, horse, cow...","[blue tit, blackbird, bluebird, crow, parrot, ...",69,38,9,60,29,"[cat, dog, elephant, giraffe, lion, horse, cow...","[pigeon, blue tit, duck, blackbird, bluebird, ...","[chicken, duck, penguin, flamingo, swan, eagle...","[cat, dog, elephant, giraffe, lion, horse, cow...","[blue tit, blackbird, bluebird, crow, parrot, ..."
1,banks_connell,animal,farm animal,"[cat, dog, elephant, giraffe, lion, horse, cow...","[cow, sheep, pig, chicken, horse, goat, duck, ...","[cat, dog, horse, cow, pig, rabbit, sheep, chi...","[elephant, giraffe, lion, tiger, fish, guinea ...","[geese, sheepdog, turkey, bull, hen, lamb, llama]",69,18,11,58,7,"[cat, dog, elephant, giraffe, lion, horse, cow...","[cow, sheep, pig, chicken, horse, goat, duck, ...","[cat, dog, horse, cow, pig, rabbit, sheep, chi...","[elephant, giraffe, lion, tiger, fish, guinea ...","[geese, sheepdog, turkey, bull, hen, lamb, llama]"


In [ ]:
# Load the smaller model for BOTH baselines.
# This makes the baseline vs SFT comparison fair.

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Loaded baseline model:", MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded baseline model: Qwen/Qwen2.5-0.5B-Instruct


In [ ]:
def generate_chat_response(messages, max_new_tokens=32, temperature=0.8, do_sample=True):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id
    }

    if do_sample:
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = 0.9
        gen_kwargs["top_k"] = 50

    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return text

In [ ]:
def build_few_shot_examples(train_rows, max_examples=4):
    """
    Build a few-shot prompt for the impostor generator.
    The impostor only sees q2 and must give one valid q2 answer.
    """
    if len(train_rows) == 0:
        return ""

    sampled = train_rows.sample(min(max_examples, len(train_rows)), random_state=SEED)

    examples = []
    for _, row in sampled.iterrows():
        target_answer = random.choice(row["q2_answers_json"])

        ex = f"""Example
Category: {row['q2_category']}
Answer: {target_answer}"""
        examples.append(ex)

    return "\n\n".join(examples)

In [ ]:
def generate_impostor_answer(q2, few_shot_text):
    """
    Generate one answer that fits q2.
    The answer should be a specific item, not a category label or explanation.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are playing a category game. "
                "You will be given one category. "
                "Give exactly ONE short answer that fits the category. "
                "Only output the answer. Do not explain."
            )
        },
        {
            "role": "user",
            "content": f"""{few_shot_text}

Now answer this round.

Category: {q2}

Answer:"""
        }
    ]

    raw_output = generate_chat_response(
        messages,
        max_new_tokens=8,
        temperature=0.6,
        do_sample=True
    )

    cleaned = clean_generated_answer(raw_output)
    cleaned = postprocess_impostor_answer(cleaned)

    return cleaned, raw_output

In [ ]:
# Words that are too abstract / generic for this game
GENERIC_BAD_ANSWERS = {
    "thing", "item", "object", "stuff", "food", "animal", "bird", "vegetable",
    "fruit", "plant", "body part", "part", "face part", "building",
    "religious building", "farm animal", "four-legged animal", "green vegetable",
    "leaf", "stem", "root", "body", "face"
}

def postprocess_impostor_answer(answer):
    """
    Convert obviously weak or invalid generations into a cleaner form if possible.
    If the answer is too generic, return it as-is for evaluation,
    but this helper gives us one place to tighten rules later.
    """
    ans = normalize_text(answer)

    # If the model outputs a long phrase, keep only the first comma-separated part
    if "," in ans:
        ans = ans.split(",")[0].strip()

    # If it outputs an article, strip it
    ans = re.sub(r"^(a|an|the)\s+", "", ans).strip()

    return ans

In [ ]:
def judge_impostor(q1, answer):
    """
    Judge whether a single answer fits the visible category q1.
    Returns 'yes' or 'no'.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a strict judge for a category game. "
                "Given a category and an answer, decide whether the answer fits the category. "
                "Reply with only 'yes' or 'no'. Do not explain."
            )
        },
        {
            "role": "user",
            "content": f"""Category: {q1}
Answer: {answer}

Does this answer fit the category? Reply only yes or no."""
        }
    ]

    raw_output = generate_chat_response(
        messages,
        max_new_tokens=3,
        temperature=0.0,
        do_sample=False
    )

    cleaned = raw_output.strip().lower()

    # normalize output (in case model says extra stuff)
    if "yes" in cleaned:
        return "yes", raw_output
    elif "no" in cleaned:
        return "no", raw_output
    else:
        return None, raw_output

In [ ]:
def classify_impostor_answer(answer, row):
    """
    Check whether the generated answer belongs to:
    - shared answers
    - q2-only answers
    - q2-valid-other
    - invalid
    Also reject generic abstract answers.
    """
    ans = normalize_text(answer)

    if ans in GENERIC_BAD_ANSWERS:
        return "invalid"

    shared = set(row["shared_answers_norm"])
    q2_only = set(row["q2_only_answers_norm"])
    q2_all = set(row["q2_answers_norm"])

    if ans in shared:
        return "shared"
    elif ans in q2_only:
        return "q2_only"
    elif ans in q2_all:
        return "q2_valid_other"
    else:
        return "invalid"

In [ ]:
import re

def clean_generated_answer(text):
    """
    Keep only the first short line and remove common formatting junk.
    """
    text = str(text).strip().split("\n")[0].strip()
    text = text.replace("Answer:", "").replace("Impostor answer:", "").strip()
    text = text.strip('"').strip("'").strip()

    # Remove leading bullets/numbers if the model adds them
    text = re.sub(r"^[\-\*\d\.\)\s]+", "", text).strip()

    return text


def postprocess_impostor_answer(answer):
    """
    Convert obviously weak or invalid generations into a cleaner form if possible.
    If the answer is too generic, return it as-is for evaluation,
    but this helper gives us one place to tighten rules later.
    """
    ans = normalize_text(answer)

    # If the model outputs a long phrase, keep only the first comma-separated part
    if "," in ans:
        ans = ans.split(",")[0].strip()

    # If it outputs an article, strip it
    ans = re.sub(r"^(a|an|the)\s+", "", ans).strip()

    return ans

In [ ]:
print("Number of category pairs:", len(df))
display(df[["q1_category", "q2_category"]])

Number of category pairs: 116


,q1_category,q2_category
0,animal,bird
1,animal,farm animal
2,animal,meat
3,animal,water bird
4,bathroom fixture,living room furniture
...,...,...
111,martial art,olympic sport
112,software application,google product
113,condiment,sandwich ingredient
114,amphibian,swamp animal


In [ ]:
def simulate_one_round(row, few_shot_text):
    """
    Simulate one round from a single category pair.
    The impostor only sees q2 and generates one q2-valid answer.
    The judge then decides whether that answer fits q1.
    """
    q1 = row["q1_category"]
    q2 = row["q2_category"]

    # Generate one impostor answer from q2 only
    impostor_answer, impostor_raw = generate_impostor_answer(q2, few_shot_text)

    # Check whether the impostor answer is valid according to q2 data
    impostor_type = classify_impostor_answer(impostor_answer, row)

    # Judge evaluates whether the answer fits q1
    judge_choice, judge_raw = judge_impostor(q1, impostor_answer)

    judge_valid = judge_choice in ["yes", "no"]

    # Ground truth: does the answer actually fit q1?
    answer_fits_q1 = (impostor_type == "shared")

    judge_correct = (
        (judge_choice == "yes" and answer_fits_q1) or
        (judge_choice == "no" and not answer_fits_q1)
    ) if judge_valid else None

    impostor_success = (judge_choice == "yes") if judge_valid else None

    return {
        "q1": q1,
        "q2": q2,
        "generated_impostor_answer": impostor_answer,
        "generated_impostor_raw": impostor_raw,
        "impostor_answer_type": impostor_type,
        "judge_choice": judge_choice,
        "judge_raw_output": judge_raw,
        "judge_valid_output": judge_valid,
        "judge_correct": judge_correct,
        "impostor_success": impostor_success,
    }

In [ ]:
for i in range(min(5, len(df))):
    test_row = df.iloc[i]

    # Use all OTHER rows as few-shot examples
    train_rows = df.drop(index=i).reset_index(drop=True)
    few_shot_text = build_few_shot_examples(train_rows, max_examples=4)

    result = simulate_one_round(test_row, few_shot_text)

    print("=" * 80)
    print("Q1:", result["q1"])
    print("Q2:", result["q2"])
    print("Generated impostor answer:", result["generated_impostor_answer"])
    print("Raw impostor output:", repr(result["generated_impostor_raw"]))
    print("Impostor answer type:", result["impostor_answer_type"])
    print("Judge choice:", result["judge_choice"])
    print("Judge raw output:", repr(result["judge_raw_output"]))
    print("Judge correct:", result["judge_correct"])

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:537: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarni

Q1: animal
Q2: bird
Generated impostor answer: pigeon
Raw impostor output: 'pigeon'
Impostor answer type: shared
Judge choice: yes
Judge raw output: 'yes'
Judge correct: True
Q1: animal
Q2: farm animal
Generated impostor answer: pig
Raw impostor output: 'pig'
Impostor answer type: shared
Judge choice: yes
Judge raw output: 'yes'
Judge correct: True


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:537: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarni

Q1: animal
Q2: meat
Generated impostor answer: pork grilled on an open fire
Raw impostor output: 'pork grilled on an open fire'
Impostor answer type: invalid
Judge choice: yes
Judge raw output: 'yes'
Judge correct: False
Q1: animal
Q2: water bird
Generated impostor answer: duck
Raw impostor output: 'duck'
Impostor answer type: shared
Judge choice: yes
Judge raw output: 'yes'
Judge correct: True
Q1: bathroom fixture
Q2: living room furniture
Generated impostor answer: coffee table
Raw impostor output: 'coffee table'
Impostor answer type: q2_only
Judge choice: yes
Judge raw output: 'yes'
Judge correct: False


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:537: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


In [ ]:
ROUNDS_PER_PAIR = 10

all_results = []

for test_idx in tqdm(range(len(df)), total=len(df)):
    test_row = df.iloc[test_idx]

    # Leave-one-pair-out few-shot setup:
    # test on one pair, use the rest as examples
    train_rows = df.drop(index=test_idx).reset_index(drop=True)
    few_shot_text = build_few_shot_examples(train_rows, max_examples=4)

    for round_num in range(ROUNDS_PER_PAIR):
        round_result = simulate_one_round(test_row, few_shot_text)
        round_result["pair_index"] = test_idx
        round_result["round_num"] = round_num
        all_results.append(round_result)

results_df = pd.DataFrame(all_results)

print("Total simulated rounds:", len(results_df))
display(results_df.head())

  0%|          | 0/116 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:537: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarni

Total simulated rounds: 1160


,q1,q2,generated_impostor_answer,generated_impostor_raw,impostor_answer_type,judge_choice,judge_raw_output,judge_valid_output,judge_correct,impostor_success,pair_index,round_num
0,animal,bird,pigeon,pigeon,shared,yes,yes,True,True,True,0,0
1,animal,bird,parrot,parrot,q2_only,yes,yes,True,False,True,0,1
2,animal,bird,parrot,parrot,q2_only,yes,yes,True,False,True,0,2
3,animal,bird,sparrow,sparrow,q2_only,yes,yes,True,False,True,0,3
4,animal,bird,parrot,parrot,q2_only,yes,yes,True,False,True,0,4


In [ ]:
valid_judge_df = results_df[results_df["judge_valid_output"] == True].copy()

valid_judge_df["blended_in"] = (valid_judge_df["judge_choice"] == "yes")
valid_judge_df["caught"] = (valid_judge_df["judge_choice"] == "no")

blended_in_rate = valid_judge_df["blended_in"].mean()
caught_rate = valid_judge_df["caught"].mean()
judge_consistency = valid_judge_df["judge_correct"].mean()

print("=== BASELINE 1 CORE METRICS ===")
print(f"Blended-in rate (impostor success): {blended_in_rate:.3f}")
print(f"Catch rate (judge said NO): {caught_rate:.3f}")

print("\n=== BASELINE 1 SECONDARY ===")
print(f"Judge consistency: {judge_consistency:.3f}")

=== BASELINE 1 CORE METRICS ===
Blended-in rate (impostor success): 0.986
Catch rate (judge said NO): 0.014

=== BASELINE 1 SECONDARY ===
Judge consistency: 0.278


In [ ]:
pair_summary = (
    valid_judge_df
    .groupby(["q1", "q2"])
    .agg(
        rounds=("round_num", "count"),
        blended_in_rate=("blended_in", "mean"),
        caught_rate=("caught", "mean"),
        judge_consistency=("judge_correct", "mean"),
    )
    .reset_index()
)

display(pair_summary)

,q1,q2,rounds,blended_in_rate,caught_rate,judge_consistency
0,amphibian,swamp animal,10,1.0,0.0,0.4
1,animal,bird,10,1.0,0.0,0.2
2,animal,farm animal,10,1.0,0.0,0.9
3,animal,meat,10,1.0,0.0,0.2
4,animal,water bird,10,1.0,0.0,1.0
...,...,...,...,...,...,...
111,watercraft,recreational vehicle,10,1.0,0.0,0.0
112,weapon,hunting gear,10,1.0,0.0,0.0
113,wild cat,jungle animal,10,1.0,0.0,0.4
114,winter sport,olympic sport,10,1.0,0.0,0.0


In [ ]:
print("=== Some blended-in answers ===")
display(
    valid_judge_df[valid_judge_df["blended_in"] == True][
        ["q1", "q2", "generated_impostor_answer", "judge_choice", "blended_in", "caught"]
    ].head(10)
)

print("\n=== Some caught answers ===")
display(
    valid_judge_df[valid_judge_df["caught"] == True][
        ["q1", "q2", "generated_impostor_answer", "judge_choice", "blended_in", "caught"]
    ].head(10)
)

=== Some blended-in answers ===


,q1,q2,generated_impostor_answer,judge_choice,blended_in,caught
0,animal,bird,pigeon,yes,True,False
1,animal,bird,parrot,yes,True,False
2,animal,bird,parrot,yes,True,False
3,animal,bird,sparrow,yes,True,False
4,animal,bird,parrot,yes,True,False
5,animal,bird,sparrow,yes,True,False
6,animal,bird,pigeon,yes,True,False
7,animal,bird,parrot,yes,True,False
8,animal,bird,sparrow,yes,True,False
9,animal,bird,parrot,yes,True,False



=== Some caught answers ===


,q1,q2,generated_impostor_answer,judge_choice,blended_in,caught
60,bird,farm animal,cow,no,False,True
183,chemical element,metal,kettle,no,False,True
900,flightless bird,antarctic animal,polar bear,no,False,True
902,flightless bird,antarctic animal,polar bear,no,False,True
903,flightless bird,antarctic animal,polar bear,no,False,True
904,flightless bird,antarctic animal,polar bear,no,False,True
905,flightless bird,antarctic animal,polar bear,no,False,True
907,flightless bird,antarctic animal,polar bear,no,False,True
908,flightless bird,antarctic animal,polar bear,no,False,True
909,flightless bird,antarctic animal,polar bear,no,False,True


In [ ]:
results_df.to_csv("impostor_round_results.csv", index=False)
pair_summary.to_csv("impostor_pair_summary.csv", index=False)

print("Saved:")
print("- impostor_round_results.csv")
print("- impostor_pair_summary.csv")

Saved:
- impostor_round_results.csv
- impostor_pair_summary.csv


In [ ]:
from google.colab import files

files.download("impostor_round_results.csv")
files.download("impostor_pair_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
results_df = pd.read_csv("impostor_round_results.csv")

print("Total rows:", len(results_df))

valid_df = results_df[results_df["judge_valid_output"] == True].copy()

valid_df["blended_in"] = (valid_df["judge_choice"] == "yes")
valid_df["caught"] = (valid_df["judge_choice"] == "no")

print("=== BASELINE 1 CORE METRICS ===")
print("Blended-in rate (impostor success):", valid_df["blended_in"].mean())
print("Catch rate (judge said NO):", valid_df["caught"].mean())

print("\n=== BASELINE 1 SECONDARY ===")
print("Judge consistency:", valid_df["judge_correct"].mean())

display(valid_df.head(10))

Total rows: 1160
=== BASELINE 1 CORE METRICS ===
Blended-in rate (impostor success): 0.9862068965517241
Catch rate (judge said NO): 0.013793103448275862

=== BASELINE 1 SECONDARY ===
Judge consistency: 0.2775862068965517


,q1,q2,generated_impostor_answer,generated_impostor_raw,impostor_answer_type,judge_choice,judge_raw_output,judge_valid_output,judge_correct,impostor_success,pair_index,round_num,blended_in,caught
0,animal,bird,pigeon,pigeon,shared,yes,yes,True,True,True,0,0,True,False
1,animal,bird,parrot,parrot,q2_only,yes,yes,True,False,True,0,1,True,False
2,animal,bird,parrot,parrot,q2_only,yes,yes,True,False,True,0,2,True,False
3,animal,bird,sparrow,sparrow,q2_only,yes,yes,True,False,True,0,3,True,False
4,animal,bird,parrot,parrot,q2_only,yes,yes,True,False,True,0,4,True,False
5,animal,bird,sparrow,sparrow,q2_only,yes,yes,True,False,True,0,5,True,False
6,animal,bird,pigeon,pigeon,shared,yes,yes,True,True,True,0,6,True,False
7,animal,bird,parrot,parrot,q2_only,yes,yes,True,False,True,0,7,True,False
8,animal,bird,sparrow,sparrow,q2_only,yes,yes,True,False,True,0,8,True,False
9,animal,bird,parrot,parrot,q2_only,yes,yes,True,False,True,0,9,True,False


In [ ]:
def build_few_shot_examples_baseline2(train_rows, max_examples=4, k_q1_examples=2):
    """
    Few-shot examples for Baseline 2.
    The model sees:
    - q2
    - a few example answers from q1
    and must output one short answer that fits q2 and blends with q1 examples.
    """
    if len(train_rows) == 0:
        return ""

    sampled = train_rows.sample(min(max_examples, len(train_rows)), random_state=SEED)

    examples = []
    for _, row in sampled.iterrows():
        q1_examples = random.sample(
            row["q1_answers_json"],
            min(k_q1_examples, len(row["q1_answers_json"]))
        )

        # Prefer shared answers if available, otherwise use any q2-valid answer
        if len(row["shared_answers_json"]) > 0:
            target_answer = random.choice(row["shared_answers_json"])
        else:
            target_answer = random.choice(row["q2_answers_json"])

        ex = f"""Example
Q2 category: {row['q2_category']}
Example answers from Q1: {", ".join(q1_examples)}
Answer: {target_answer}"""
        examples.append(ex)

    return "\n\n".join(examples)

In [ ]:
def generate_impostor_answer_baseline2(q2, q1_example_answers, few_shot_text):
    """
    Baseline 2:
    The impostor sees q2 plus a few example answers from q1.
    It tries to give one q2-valid answer that is close to those q1 examples.
    """
    q1_examples_text = ", ".join(q1_example_answers)

    messages = [
        {
            "role": "system",
            "content": (
                "You are the impostor in a category game. "
                "You will be given your category (Q2) and a few example answers from another category (Q1). "
                "Your job is to give exactly one short concrete answer that fits Q2 "
                "and is as similar as possible to the Q1 example answers. "
                "Only output the answer. Do not explain."
            )
        },
        {
            "role": "user",
            "content": f"""{few_shot_text}

Now answer this round.

Q2 category: {q2}
Example answers from Q1: {q1_examples_text}

Answer:"""
        }
    ]

    raw_output = generate_chat_response(
        messages,
        max_new_tokens=8,
        temperature=0.8,
        do_sample=True
    )

    cleaned = clean_generated_answer(raw_output)
    cleaned = postprocess_impostor_answer(cleaned)

    return cleaned, raw_output

In [ ]:
def simulate_one_round_baseline2(row, few_shot_text, k_q1_examples=2):
    """
    Simulate one round for Baseline 2.
    The impostor sees q2 and k example answers from q1.
    """
    q1 = row["q1_category"]
    q2 = row["q2_category"]

    q1_example_answers = random.sample(
        row["q1_answers_json"],
        min(k_q1_examples, len(row["q1_answers_json"]))
    )

    impostor_answer, impostor_raw = generate_impostor_answer_baseline2(
        q2=q2,
        q1_example_answers=q1_example_answers,
        few_shot_text=few_shot_text
    )

    impostor_type = classify_impostor_answer(impostor_answer, row)

    judge_choice, judge_raw = judge_impostor(q1, impostor_answer)

    judge_valid = judge_choice in ["yes", "no"]

    answer_fits_q1 = (impostor_type == "shared")

    judge_correct = (
        (judge_choice == "yes" and answer_fits_q1) or
        (judge_choice == "no" and not answer_fits_q1)
    ) if judge_valid else None

    blended_in = (judge_choice == "yes") if judge_valid else None
    caught_by_judge = (judge_choice == "no") if judge_valid else None

    return {
        "q1": q1,
        "q2": q2,
        "q1_example_answers": q1_example_answers,
        "generated_impostor_answer": impostor_answer,
        "generated_impostor_raw": impostor_raw,
        "impostor_answer_type": impostor_type,
        "judge_choice": judge_choice,
        "judge_raw_output": judge_raw,
        "judge_valid_output": judge_valid,
        "judge_correct": judge_correct,
        "blended_in": blended_in,
        "caught_by_judge": caught_by_judge,
    }

In [ ]:
for i in range(min(5, len(df))):
    test_row = df.iloc[i]

    train_rows = df.drop(index=i).reset_index(drop=True)
    few_shot_text = build_few_shot_examples_baseline2(
        train_rows,
        max_examples=4,
        k_q1_examples=2
    )

    result = simulate_one_round_baseline2(
        test_row,
        few_shot_text,
        k_q1_examples=2
    )

    print("=" * 80)
    print("Q1:", result["q1"])
    print("Q2:", result["q2"])
    print("Q1 example answers shown to impostor:", result["q1_example_answers"])
    print("Generated impostor answer:", result["generated_impostor_answer"])
    print("Raw impostor output:", repr(result["generated_impostor_raw"]))
    print("Impostor answer type:", result["impostor_answer_type"])
    print("Judge choice:", result["judge_choice"])
    print("Judge raw output:", repr(result["judge_raw_output"]))
    print("Judge correct:", result["judge_correct"])
    print("Blended in:", result["blended_in"])

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:537: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarni

Q1: animal
Q2: bird
Q1 example answers shown to impostor: ['lizard', 'frog']
Generated impostor answer: frog
Raw impostor output: 'frog'
Impostor answer type: invalid
Judge choice: yes
Judge raw output: 'yes'
Judge correct: False
Blended in: True
Q1: animal
Q2: farm animal
Q1 example answers shown to impostor: ['whale', 'dolphin']
Generated impostor answer: dolphin
Raw impostor output: 'dolphin'
Impostor answer type: invalid
Judge choice: yes
Judge raw output: 'yes'
Judge correct: False
Blended in: True


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:537: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarni

Q1: animal
Q2: meat
Q1 example answers shown to impostor: ['hedgehog', 'kangaroo']
Generated impostor answer: chicken
Raw impostor output: 'chicken'
Impostor answer type: shared
Judge choice: yes
Judge raw output: 'yes'
Judge correct: True
Blended in: True
Q1: animal
Q2: water bird
Q1 example answers shown to impostor: ['raccoon', 'lizard']
Generated impostor answer: turtle
Raw impostor output: 'turtle'
Impostor answer type: invalid
Judge choice: yes
Judge raw output: 'yes'
Judge correct: False
Blended in: True
Q1: bathroom fixture
Q2: living room furniture
Q1 example answers shown to impostor: ['sink', 'door']
Generated impostor answer: chair
Raw impostor output: 'chair'
Impostor answer type: q2_only
Judge choice: yes
Judge raw output: 'yes'
Judge correct: False
Blended in: True


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:537: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


In [ ]:
ROUNDS_PER_PAIR = 5
K_Q1_EXAMPLES = 2

baseline2_results = []

for pair_index in range(len(df)):
    test_row = df.iloc[pair_index]
    train_rows = df.drop(index=pair_index).reset_index(drop=True)

    few_shot_text = build_few_shot_examples_baseline2(
        train_rows,
        max_examples=4,
        k_q1_examples=K_Q1_EXAMPLES
    )

    for round_num in range(ROUNDS_PER_PAIR):
        result = simulate_one_round_baseline2(
            test_row,
            few_shot_text,
            k_q1_examples=K_Q1_EXAMPLES
        )
        result["pair_index"] = pair_index
        result["round_num"] = round_num
        baseline2_results.append(result)

baseline2_results_df = pd.DataFrame(baseline2_results)
print("Total rows:", len(baseline2_results_df))
display(baseline2_results_df.head(10))

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:537: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarni

Total rows: 580


,q1,q2,q1_example_answers,generated_impostor_answer,generated_impostor_raw,impostor_answer_type,judge_choice,judge_raw_output,judge_valid_output,judge_correct,blended_in,caught_by_judge,pair_index,round_num
0,animal,bird,"[hamster, goose]",duck,duck,shared,yes,yes,True,True,True,False,0,0
1,animal,bird,"[shark, bird]",shark,shark,invalid,yes,yes,True,False,True,False,0,1
2,animal,bird,"[seagull, jaguar]",seagull,seagull,shared,yes,yes,True,True,True,False,0,2
3,animal,bird,"[cow, gerbil]",snake,snake,invalid,yes,yes,True,False,True,False,0,3
4,animal,bird,"[rat, insect]",rat,rat,invalid,yes,yes,True,False,True,False,0,4
5,animal,farm animal,"[goose, human]",hedgehog,hedgehog,invalid,yes,yes,True,False,True,False,1,0
6,animal,farm animal,"[mouse, koala]",cow,cow,shared,yes,yes,True,True,True,False,1,1
7,animal,farm animal,"[human, squirrel]",horse,horse,shared,yes,yes,True,True,True,False,1,2
8,animal,farm animal,"[gerbil, badger]",dog,dog,shared,yes,yes,True,True,True,False,1,3
9,animal,farm animal,"[seagull, panda]",cattle,cattle,invalid,yes,yes,True,False,True,False,1,4


In [ ]:
valid_b2_df = baseline2_results_df[baseline2_results_df["judge_valid_output"] == True].copy()

valid_b2_df["blended_in"] = (valid_b2_df["judge_choice"] == "yes")
valid_b2_df["caught"] = (valid_b2_df["judge_choice"] == "no")

blended_in_rate_b2 = valid_b2_df["blended_in"].mean()
caught_rate_b2 = valid_b2_df["caught"].mean()
judge_consistency_b2 = valid_b2_df["judge_correct"].mean()

print("=== BASELINE 2 CORE METRICS ===")
print(f"Blended-in rate (impostor success): {blended_in_rate_b2:.3f}")
print(f"Catch rate (judge said NO): {caught_rate_b2:.3f}")

print("\n=== BASELINE 2 SECONDARY ===")
print(f"Judge consistency: {judge_consistency_b2:.3f}")

=== BASELINE 2 CORE METRICS ===
Blended-in rate (impostor success): 0.990
Catch rate (judge said NO): 0.010

=== BASELINE 2 SECONDARY ===
Judge consistency: 0.316


In [ ]:
baseline2_pair_summary = (
    valid_b2_df
    .groupby(["q1", "q2"])
    .agg(
        rounds=("round_num", "count"),
        blended_in_rate=("blended_in", "mean"),
        caught_rate=("caught", "mean"),
        judge_consistency=("judge_correct", "mean"),
    )
    .reset_index()
)

display(baseline2_pair_summary)

,q1,q2,rounds,blended_in_rate,caught_rate,judge_consistency
0,amphibian,swamp animal,5,1.0,0.0,0.6
1,animal,bird,5,1.0,0.0,0.4
2,animal,farm animal,5,1.0,0.0,0.6
3,animal,meat,5,1.0,0.0,0.4
4,animal,water bird,5,1.0,0.0,0.6
...,...,...,...,...,...,...
111,watercraft,recreational vehicle,5,1.0,0.0,0.2
112,weapon,hunting gear,5,1.0,0.0,0.6
113,wild cat,jungle animal,5,1.0,0.0,0.4
114,winter sport,olympic sport,5,1.0,0.0,0.0


In [ ]:
print("=== Baseline 2 blended-in answers ===")
display(
    valid_b2_df[valid_b2_df["blended_in"] == True][
        ["q1", "q2", "q1_example_answers", "generated_impostor_answer", "judge_choice", "blended_in", "caught"]
    ].head(10)
)

print("\n=== Baseline 2 caught answers ===")
display(
    valid_b2_df[valid_b2_df["caught"] == True][
        ["q1", "q2", "q1_example_answers", "generated_impostor_answer", "judge_choice", "blended_in", "caught"]
    ].head(10)
)

=== Baseline 2 blended-in answers ===


,q1,q2,q1_example_answers,generated_impostor_answer,judge_choice,blended_in,caught
0,animal,bird,"[hamster, goose]",duck,yes,True,False
1,animal,bird,"[shark, bird]",shark,yes,True,False
2,animal,bird,"[seagull, jaguar]",seagull,yes,True,False
3,animal,bird,"[cow, gerbil]",snake,yes,True,False
4,animal,bird,"[rat, insect]",rat,yes,True,False
5,animal,farm animal,"[goose, human]",hedgehog,yes,True,False
6,animal,farm animal,"[mouse, koala]",cow,yes,True,False
7,animal,farm animal,"[human, squirrel]",horse,yes,True,False
8,animal,farm animal,"[gerbil, badger]",dog,yes,True,False
9,animal,farm animal,"[seagull, panda]",cattle,yes,True,False



=== Baseline 2 caught answers ===


,q1,q2,q1_example_answers,generated_impostor_answer,judge_choice,blended_in,caught
39,bird,meat,"[jackdaw, bluebird]",veal,no,False,True
354,reptile,pet,"[crocodile, lizard]",dog,no,False,True
486,bird of prey,scavenger,"[kestrel, condor]",raccoon,no,False,True
488,bird of prey,scavenger,"[kestrel, falcon]",snail,no,False,True
489,bird of prey,scavenger,"[falcon, kite]",mole,no,False,True
555,martial art,olympic sport,"[judo, karate]",basketball,no,False,True


In [ ]:
baseline2_results_df.to_csv("impostor_round_results_baseline2.csv", index=False)
baseline2_pair_summary.to_csv("impostor_pair_summary_baseline2.csv", index=False)

print("Saved:")
print("- impostor_round_results_baseline2.csv")
print("- impostor_pair_summary_baseline2.csv")

Saved:
- impostor_round_results_baseline2.csv
- impostor_pair_summary_baseline2.csv


In [ ]:
from google.colab import files

files.download("impostor_round_results_baseline2.csv")
files.download("impostor_pair_summary_baseline2.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
baseline2_results_df = pd.read_csv("impostor_round_results_baseline2.csv")

print("Total rows:", len(baseline2_results_df))

valid_b2_df = baseline2_results_df[baseline2_results_df["judge_valid_output"] == True].copy()

valid_b2_df["blended_in"] = (valid_b2_df["judge_choice"] == "yes")
valid_b2_df["caught"] = (valid_b2_df["judge_choice"] == "no")

print("=== BASELINE 2 CORE METRICS ===")
print("Blended-in rate (impostor success):", valid_b2_df["blended_in"].mean())
print("Catch rate (judge said NO):", valid_b2_df["caught"].mean())

print("\n=== BASELINE 2 SECONDARY ===")
print("Judge consistency:", valid_b2_df["judge_correct"].mean())

display(valid_b2_df.head(10))

Total rows: 580
=== BASELINE 2 CORE METRICS ===
Blended-in rate (impostor success): 0.9896551724137931
Catch rate (judge said NO): 0.010344827586206896

=== BASELINE 2 SECONDARY ===
Judge consistency: 0.31551724137931036


,q1,q2,q1_example_answers,generated_impostor_answer,generated_impostor_raw,impostor_answer_type,judge_choice,judge_raw_output,judge_valid_output,judge_correct,blended_in,caught_by_judge,pair_index,round_num,caught
0,animal,bird,"['hamster', 'goose']",duck,duck,shared,yes,yes,True,True,True,False,0,0,False
1,animal,bird,"['shark', 'bird']",shark,shark,invalid,yes,yes,True,False,True,False,0,1,False
2,animal,bird,"['seagull', 'jaguar']",seagull,seagull,shared,yes,yes,True,True,True,False,0,2,False
3,animal,bird,"['cow', 'gerbil']",snake,snake,invalid,yes,yes,True,False,True,False,0,3,False
4,animal,bird,"['rat', 'insect']",rat,rat,invalid,yes,yes,True,False,True,False,0,4,False
5,animal,farm animal,"['goose', 'human']",hedgehog,hedgehog,invalid,yes,yes,True,False,True,False,1,0,False
6,animal,farm animal,"['mouse', 'koala']",cow,cow,shared,yes,yes,True,True,True,False,1,1,False
7,animal,farm animal,"['human', 'squirrel']",horse,horse,shared,yes,yes,True,True,True,False,1,2,False
8,animal,farm animal,"['gerbil', 'badger']",dog,dog,shared,yes,yes,True,True,True,False,1,3,False
9,animal,farm animal,"['seagull', 'panda']",cattle,cattle,invalid,yes,yes,True,False,True,False,1,4,False


In [ ]:
def build_sft_examples(df, k_q1_examples=2, examples_per_row=3):
    """
    Build prompt/completion training examples for SFT.

    Prompt:
      - q2 category
      - a few example answers from q1

    Completion:
      - a good camouflage answer, preferring shared answers when possible
    """
    sft_rows = []

    for _, row in df.iterrows():
        q1_answers = row["q1_answers_json"]
        q2_answers = row["q2_answers_json"]
        shared_answers = row["shared_answers_json"]

        if len(q1_answers) == 0 or len(q2_answers) == 0:
            continue

        for _ in range(examples_per_row):
            q1_example_answers = random.sample(
                q1_answers,
                min(k_q1_examples, len(q1_answers))
            )

            # Prefer shared answers for camouflage training
            if len(shared_answers) > 0:
                target_answer = random.choice(shared_answers)
            else:
                target_answer = random.choice(q2_answers)

            prompt = f"""You are the impostor in a category game.
You will be given your category (Q2) and a few example answers from the visible category (Q1).
Give exactly one short concrete answer that fits Q2 and is as similar as possible to the Q1 example answers.
Only output the answer. Do not explain.

Q2 category: {row['q2_category']}
Example answers from Q1: {", ".join(q1_example_answers)}

Answer:"""

            completion = target_answer

            sft_rows.append({
                "prompt": prompt,
                "completion": completion,
                "q1": row["q1_category"],
                "q2": row["q2_category"],
                "q1_example_answers": q1_example_answers,
                "target_answer": target_answer,
            })

    return pd.DataFrame(sft_rows)

In [ ]:
sft_df = build_sft_examples(df, k_q1_examples=2, examples_per_row=5)

print("Total SFT training examples:", len(sft_df))
display(sft_df.head(10))

Total SFT training examples: 580


,prompt,completion,q1,q2,q1_example_answers,target_answer
0,You are the impostor in a category game.\nYou ...,swan,animal,bird,"[chimpanzee, tiger]",swan
1,You are the impostor in a category game.\nYou ...,goose,animal,bird,"[mouse, fox]",goose
2,You are the impostor in a category game.\nYou ...,penguin,animal,bird,"[lizard, camel]",penguin
3,You are the impostor in a category game.\nYou ...,eagle,animal,bird,"[sheep, polar bear]",eagle
4,You are the impostor in a category game.\nYou ...,goose,animal,bird,"[cow, guinea pig]",goose
5,You are the impostor in a category game.\nYou ...,rabbit,animal,farm animal,"[ferret, tiger]",rabbit
6,You are the impostor in a category game.\nYou ...,cat,animal,farm animal,"[chimpanzee, kangaroo]",cat
7,You are the impostor in a category game.\nYou ...,chicken,animal,farm animal,"[ant, human]",chicken
8,You are the impostor in a category game.\nYou ...,donkey,animal,farm animal,"[ferret, swan]",donkey
9,You are the impostor in a category game.\nYou ...,horse,animal,farm animal,"[bat, hedgehog]",horse


In [ ]:
sft_df.to_csv("impostor_sft_dataset.csv", index=False)
print("Saved: impostor_sft_dataset.csv")

Saved: impostor_sft_dataset.csv


In [ ]:
from google.colab import files
files.download("impostor_sft_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    sft_df[["prompt", "completion"]].reset_index(drop=True)
)

print(train_dataset)
print(train_dataset[0])

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 580
})
{'prompt': 'You are the impostor in a category game.\nYou will be given your category (Q2) and a few example answers from the visible category (Q1).\nGive exactly one short concrete answer that fits Q2 and is as similar as possible to the Q1 example answers.\nOnly output the answer. Do not explain.\n\nQ2 category: bird\nExample answers from Q1: chimpanzee, tiger\n\nAnswer:', 'completion': 'swan'}


In [ ]:
# Load a smaller model for SFT training.
# We switch to a smaller instruct model because Phi-3 mini is too heavy
# for this Colab RAM setup during training.

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Disable cache during training.
model.config.use_cache = False

print("Loaded model for SFT training:", MODEL_NAME)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loaded model for SFT training: Qwen/Qwen2.5-0.5B-Instruct


In [ ]:
# LoRA adapter configuration.
# This trains small adapter weights instead of the full model.

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)

In [ ]:
# Training settings for SFT.
# These are made lighter for Colab memory.

training_args = SFTConfig(
    output_dir="small-impostor-sft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="no",
    fp16=False,
    report_to="none",
)

# Build the SFT trainer.
# max_seq_length keeps prompts shorter, which helps memory.

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    peft_config=peft_config,
    max_seq_length=192,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:269: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:355: UserWarning: You passed a `dataset_kwargs` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/580 [00:00<?, ? examples/s]

In [ ]:
# Start supervised fine-tuning.
trainer.train()

Step,Training Loss
5,3.670000
10,3.115000
15,2.612500
20,2.188800
25,1.766900
30,1.300400
35,0.843800
40,0.525400
45,0.424600
50,0.371900


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


TrainOutput(global_step=435, training_loss=0.4041746054572621, metrics={'train_runtime': 216.8598, 'train_samples_per_second': 8.024, 'train_steps_per_second': 2.006, 'total_flos': 443658321773568.0, 'train_loss': 0.4041746054572621, 'epoch': 3.0})

In [ ]:
# Save the trained LoRA adapter and tokenizer files.
# This lets you reload the fine-tuned model later without retraining.

trainer.model.save_pretrained("impostor-sft-adapter")
tokenizer.save_pretrained("impostor-sft-adapter")

print("Saved adapter to: impostor-sft-adapter")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Saved adapter to: impostor-sft-adapter


In [ ]:
# Reload the base model for inference.
# Then attach the trained LoRA adapter on top of it.

SFT_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

sft_tokenizer = AutoTokenizer.from_pretrained("impostor-sft-adapter")
sft_tokenizer.padding_side = "right"

if sft_tokenizer.pad_token is None:
    sft_tokenizer.pad_token = sft_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

sft_model = PeftModel.from_pretrained(base_model, "impostor-sft-adapter")
sft_model.eval()

print("Loaded SFT model for inference.")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loaded SFT model for inference.


In [ ]:
# Generate text using the fine-tuned SFT model.
# This is the same idea as your earlier generation helper,
# but it uses sft_model instead of the baseline model.

def generate_chat_response_sft(messages, max_new_tokens=32, temperature=0.8, do_sample=True):
    prompt = sft_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = sft_tokenizer(prompt, return_tensors="pt").to(sft_model.device)

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": sft_tokenizer.eos_token_id,
        "eos_token_id": sft_tokenizer.eos_token_id,
    }

    if do_sample:
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = 0.95

    with torch.no_grad():
        outputs = sft_model.generate(**inputs, **gen_kwargs)

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    text = sft_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return text

In [ ]:
# Generate one impostor answer using the fine-tuned SFT model.
# This uses the same Baseline 2 task format:
# q2 + example answers from q1 -> one short answer.

def generate_impostor_answer_sft(q2, q1_example_answers, few_shot_text):
    q1_examples_text = ", ".join(q1_example_answers)

    messages = [
        {
            "role": "system",
            "content": (
                "You are the impostor in a category game. "
                "You will be given your category (Q2) and a few example answers from another category (Q1). "
                "Your job is to give exactly one short concrete answer that fits Q2 "
                "and is as similar as possible to the Q1 example answers. "
                "Only output the answer. Do not explain."
            )
        },
        {
            "role": "user",
            "content": f"""{few_shot_text}

Now answer this round.

Q2 category: {q2}
Example answers from Q1: {q1_examples_text}

Answer:"""
        }
    ]

    raw_output = generate_chat_response_sft(
        messages,
        max_new_tokens=8,
        temperature=0.8,
        do_sample=True
    )

    cleaned = clean_generated_answer(raw_output)
    cleaned = postprocess_impostor_answer(cleaned)

    return cleaned, raw_output

In [ ]:
# Run one round using the fine-tuned SFT impostor model.
# The judge stays the same as before.

def simulate_one_round_sft(row, few_shot_text, k_q1_examples=2):
    q1 = row["q1_category"]
    q2 = row["q2_category"]

    q1_example_answers = random.sample(
        row["q1_answers_json"],
        min(k_q1_examples, len(row["q1_answers_json"]))
    )

    impostor_answer, impostor_raw = generate_impostor_answer_sft(
        q2=q2,
        q1_example_answers=q1_example_answers,
        few_shot_text=few_shot_text
    )

    judge_choice, judge_raw = judge_impostor(q1, impostor_answer)
    judge_valid = judge_choice in ["yes", "no"]

    blended_in = (judge_choice == "yes") if judge_valid else None
    caught = (judge_choice == "no") if judge_valid else None

    return {
        "q1": q1,
        "q2": q2,
        "q1_example_answers": q1_example_answers,
        "generated_impostor_answer": impostor_answer,
        "generated_impostor_raw": impostor_raw,
        "judge_choice": judge_choice,
        "judge_raw_output": judge_raw,
        "judge_valid_output": judge_valid,
        "blended_in": blended_in,
        "caught": caught,
    }

In [ ]:
# Quick sanity check on a few examples before running the full evaluation.

for i in range(min(5, len(df))):
    test_row = df.iloc[i]

    train_rows = df.drop(index=i).reset_index(drop=True)
    few_shot_text = build_few_shot_examples_baseline2(
        train_rows,
        max_examples=4,
        k_q1_examples=2
    )

    result = simulate_one_round_sft(
        test_row,
        few_shot_text,
        k_q1_examples=2
    )

    print("=" * 80)
    print("Q1:", result["q1"])
    print("Q2:", result["q2"])
    print("Q1 example answers shown to impostor:", result["q1_example_answers"])
    print("Generated SFT impostor answer:", result["generated_impostor_answer"])
    print("Raw SFT impostor output:", repr(result["generated_impostor_raw"]))
    print("Judge choice:", result["judge_choice"])
    print("Blended in:", result["blended_in"])
    print("Caught:", result["caught"])

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:537: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Q1: animal
Q2: bird
Q1 example answers shown to impostor: ['fish', 'goose']
Generated SFT impostor answer: swan
Raw SFT impostor output: 'swan'
Judge choice: yes
Blended in: True
Caught: False
Q1: animal
Q2: farm animal
Q1 example answers shown to impostor: ['goat', 'shark']
Generated SFT impostor answer: horse
Raw SFT impostor output: 'horse'
Judge choice: yes
Blended in: True
Caught: False


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:537: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarni

Q1: animal
Q2: meat
Q1 example answers shown to impostor: ['hamster', 'fish']
Generated SFT impostor answer: duck
Raw SFT impostor output: 'duck'
Judge choice: yes
Blended in: True
Caught: False
Q1: animal
Q2: water bird
Q1 example answers shown to impostor: ['butterfly', 'antelope']
Generated SFT impostor answer: sparrow
Raw SFT impostor output: 'sparrow'
Judge choice: yes
Blended in: True
Caught: False


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:537: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarni

Q1: bathroom fixture
Q2: living room furniture
Q1 example answers shown to impostor: ['bidet', 'door']
Generated SFT impostor answer: sink
Raw SFT impostor output: 'sink'
Judge choice: yes
Blended in: True
Caught: False


In [ ]:
# Evaluate the fine-tuned model on the whole dataset.

ROUNDS_PER_PAIR = 5
K_Q1_EXAMPLES = 2

sft_results = []

for pair_index in range(len(df)):
    test_row = df.iloc[pair_index]
    train_rows = df.drop(index=pair_index).reset_index(drop=True)

    few_shot_text = build_few_shot_examples_baseline2(
        train_rows,
        max_examples=4,
        k_q1_examples=K_Q1_EXAMPLES
    )

    for round_num in range(ROUNDS_PER_PAIR):
        result = simulate_one_round_sft(
            test_row,
            few_shot_text,
            k_q1_examples=K_Q1_EXAMPLES
        )
        result["pair_index"] = pair_index
        result["round_num"] = round_num
        sft_results.append(result)

sft_results_df = pd.DataFrame(sft_results)

print("Total SFT evaluation rows:", len(sft_results_df))
display(sft_results_df.head(10))

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:537: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarni

Total SFT evaluation rows: 580


,q1,q2,q1_example_answers,generated_impostor_answer,generated_impostor_raw,judge_choice,judge_raw_output,judge_valid_output,blended_in,caught,pair_index,round_num
0,animal,bird,"[snake, wolf]",skunk,skunk,yes,yes,True,True,False,0,0
1,animal,bird,"[gerbil, mouse]",skunk,skunk,yes,yes,True,True,False,0,1
2,animal,bird,"[jaguar, bat]",jaguar,jaguar,yes,yes,True,True,False,0,2
3,animal,bird,"[ant, shark]",snake,snake,yes,yes,True,True,False,0,3
4,animal,bird,"[kangaroo, pig]",snake,snake,yes,yes,True,True,False,0,4
5,animal,farm animal,"[chimpanzee, insect]",dog,dog,yes,yes,True,True,False,1,0
6,animal,farm animal,"[cheetah, squirrel]",cow,cow,yes,yes,True,True,False,1,1
7,animal,farm animal,"[zebra, gerbil]",horse,horse,yes,yes,True,True,False,1,2
8,animal,farm animal,"[wolf, squirrel]",horse,horse,yes,yes,True,True,False,1,3
9,animal,farm animal,"[goose, eagle]",cow,cow,yes,yes,True,True,False,1,4


In [ ]:
# Compute the main SFT evaluation metrics.
# These are the same simple metrics you used for the baselines.

valid_sft_df = sft_results_df[sft_results_df["judge_valid_output"] == True].copy()

valid_sft_df["blended_in"] = (valid_sft_df["judge_choice"] == "yes")
valid_sft_df["caught"] = (valid_sft_df["judge_choice"] == "no")

print("=== SFT CORE METRICS ===")
print("Blended-in rate (impostor success):", valid_sft_df["blended_in"].mean())
print("Catch rate (judge said NO):", valid_sft_df["caught"].mean())

display(valid_sft_df.head(10))

=== SFT CORE METRICS ===
Blended-in rate (impostor success): 1.0
Catch rate (judge said NO): 0.0


,q1,q2,q1_example_answers,generated_impostor_answer,generated_impostor_raw,judge_choice,judge_raw_output,judge_valid_output,blended_in,caught,pair_index,round_num
0,animal,bird,"[snake, wolf]",skunk,skunk,yes,yes,True,True,False,0,0
1,animal,bird,"[gerbil, mouse]",skunk,skunk,yes,yes,True,True,False,0,1
2,animal,bird,"[jaguar, bat]",jaguar,jaguar,yes,yes,True,True,False,0,2
3,animal,bird,"[ant, shark]",snake,snake,yes,yes,True,True,False,0,3
4,animal,bird,"[kangaroo, pig]",snake,snake,yes,yes,True,True,False,0,4
5,animal,farm animal,"[chimpanzee, insect]",dog,dog,yes,yes,True,True,False,1,0
6,animal,farm animal,"[cheetah, squirrel]",cow,cow,yes,yes,True,True,False,1,1
7,animal,farm animal,"[zebra, gerbil]",horse,horse,yes,yes,True,True,False,1,2
8,animal,farm animal,"[wolf, squirrel]",horse,horse,yes,yes,True,True,False,1,3
9,animal,farm animal,"[goose, eagle]",cow,cow,yes,yes,True,True,False,1,4
